In [2]:
from langchain_ollama import ChatOllama

base_url = 'http://localhost:11434'
model = 'qwen3:0.6b'

llm = ChatOllama(base_url=base_url, model=model)
llm

ChatOllama(model='qwen3:0.6b', base_url='http://localhost:11434')

## Sequential LCEL Chain

In [3]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate

system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate(messages)

question = template.invoke({'school': 'primary', 'topics': 'solar system', 'points': 5})

response = llm.invoke(question)
print(response.content)

1. The solar system is a galaxy of planets, sun, and other celestial bodies.  
2. Planets orbit the sun in a spiral path.  
3. Mercury is the closest planet to the sun, followed by Venus.  
4. The moon orbits the Earth, a satellite of the planet.  
5. Planets vary in size and distance from the sun, with Earth being the closest.


In [4]:
system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')
question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate(messages)

chain = template | llm

In [5]:
response = chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 5})
print(response.content)

1. The solar system is a star with eight planets, including Earth.  
2. Planets orbit the Sun in elliptical paths.  
3. The planets are in order from the Sun: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.  
4. The Sun’s mass and radius are critical for planetary motion.  
5. Space exploration has expanded our understanding of the solar system’s structure and history.


In [6]:
response = chain.invoke({'school': 'phd', 'topics': 'solar system', 'points': 5})
print(response.content)

1. The **Solar System** is a **massive star system** with the Sun at its center, surrounded by planets, moons, and other celestial bodies.  
2. Planets orbit the Sun in **order** from the closest (Mercury) to the farthest (Jupiter), each with unique characteristics.  
3. The **Moon** orbits the Earth, formed from **comets and asteroids**, and remains a **natural satellite** due to tidal locking.  
4. Beyond Neptune, there are **dwarf planets** like Pluto and **asteroids**, as well as the **Kuiper Belt** and **Trojan asteroids**.  
5. The **Sun** generates energy that powers all life on Earth, making it a **central celestial body**.


In [7]:
response

AIMessage(content='1. The **Solar System** is a **massive star system** with the Sun at its center, surrounded by planets, moons, and other celestial bodies.  \n2. Planets orbit the Sun in **order** from the closest (Mercury) to the farthest (Jupiter), each with unique characteristics.  \n3. The **Moon** orbits the Earth, formed from **comets and asteroids**, and remains a **natural satellite** due to tidal locking.  \n4. Beyond Neptune, there are **dwarf planets** like Pluto and **asteroids**, as well as the **Kuiper Belt** and **Trojan asteroids**.  \n5. The **Sun** generates energy that powers all life on Earth, making it a **central celestial body**.', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-03-05T04:50:31.70427Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3671199250, 'load_duration': 54144000, 'prompt_eval_count': 37, 'prompt_eval_duration': 29003917, 'eval_count': 370, 'eval_duration': 3243658841, 'logprobs': None, 'model_n

In [8]:
from langchain_core.output_parsers import StrOutputParser

In [9]:
chain = template | llm | StrOutputParser()
response = chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 5})
print(response)

1. The solar system is our galaxy, with the Sun at the center.  
2. It includes the Sun, planets like Earth, and other celestial bodies.  
3. Planets orbit the Sun in elliptical paths.  
4. The Moon orbits the Earth, forming a natural satellite.  
5. The Sun generates energy that powers the solar system.


In [10]:
response

'1. The solar system is our galaxy, with the Sun at the center.  \n2. It includes the Sun, planets like Earth, and other celestial bodies.  \n3. Planets orbit the Sun in elliptical paths.  \n4. The Moon orbits the Earth, forming a natural satellite.  \n5. The Sun generates energy that powers the solar system.'

## Chaining Runnables (Chain Multiple Runnables)
- We can even combine this chain with more runnables to create another chain.
- Let's see how easy our generated output is?

In [12]:
chain

ChatPromptTemplate(input_variables=['points', 'school', 'topics'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['school'], input_types={}, partial_variables={}, template='You are {school} teacher. You answer in short sentences.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['points', 'topics'], input_types={}, partial_variables={}, template='tell me about the {topics} in {points} points'), additional_kwargs={})])
| ChatOllama(model='qwen3:0.6b', base_url='http://localhost:11434')
| StrOutputParser()

In [13]:
analysis_prompt = ChatPromptTemplate.from_template("""analyze the following text: {response}
                                                    You need tell me that how difficult it is to understand.
                                                    Answer in one sentence only.
                                                   """)
fact_check_chain = analysis_prompt | llm | StrOutputParser()
output = fact_check_chain.invoke({'response': response})
print(output)

Understanding the solar system requires recognizing its structure as a galaxy with the Sun at its center, planets in elliptical orbits, the Moon as a natural satellite, and the Sun's role in generating energy that powers the system.


In [14]:
composed_chain = {"response": chain} | analysis_prompt | llm | StrOutputParser()

output = composed_chain.invoke({'school': 'phd', 'topics': 'solar system', 'points': 5})
print(output)

The solar system's complexity, including its structure and celestial bodies, makes it challenging to fully grasp.


## Parallel LCEL Chain
- Parallel chains are used to run multiple runnables in parallel.
- The final return value is a dict with the results of each value under its appropriate key.

In [15]:
system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')
question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate(messages)
fact_chain = template | llm | StrOutputParser()

output = fact_chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 2})
print(output)

1. The solar system consists of the Sun and eight planets, including Earth, orbiting around the Sun.  
2. Planets like Mercury and Pluto orbit the Sun in distinct paths, while Earth's orbit is the closest and most stable.


In [16]:
question = HumanMessagePromptTemplate.from_template('write a poem on {topics} in {sentences} lines')

messages = [system, question]
template = ChatPromptTemplate(messages)
poem_chain = template | llm | StrOutputParser()

output = poem_chain.invoke({'school': 'primary', 'topics': 'solar system', 'sentences': 2})
print(output)

The sun hums its light, a beacon through the night,  
Or planets orbit like stars in a dance.


In [17]:
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel(fact = fact_chain, poem = poem_chain)

In [18]:
output = chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 2, 'sentences': 2})
print(output['fact'])
print('\n\n')
print(output['poem'])

1. The solar system consists of the Sun and eight major planets, including Earth, orbiting the Sun.  
2. Planets like Mercury, Venus, and Mars orbit the Sun in elliptical paths, with Earth being the closest and the largest in size.



The Sun radiates a golden light, a beacon of our home.  
Earth's orbit, a circle of dreams, we gaze.


## Chain Router
- The router chain is used to route the output of a previous runnable to the next runnable based on the output of the previous runnable.

In [20]:
prompt = """Given the user review below, classify it as either being about `Positive` or `Negative`.
            Do not respond with more than one word.

            Review: {review}
            Classification:"""
template = ChatPromptTemplate.from_template(prompt)

chain = template | llm | StrOutputParser()

review = "Thank you so much for provideing such a great plateform for learning. I am really happy with the service."
# review = "I am not happy with service. It is not good."
chain.invoke({'review': review})


'Positive'

In [21]:
positive_prompt = """
                    You are expert in writing reply for positive reviews.
                    You need to encourage the user to share their experience on social media.
                    Review: {review}
                    Answer:"""
positive_template = ChatPromptTemplate.from_template(positive_prompt)
positive_chain = positive_template | llm | StrOutputParser()

In [28]:
negative_prompt = """
                    You are expert in writing reply for negative reviews.
                    You need first to apologize for the inconvenience caused to the user.
                    You need to encourage the user to share their concern on following Email:'example@email.com'.
                    Review: {review}
                    Answer:"""

negative_template = ChatPromptTemplate.from_template(negative_prompt)
negative_chain = negative_template | llm | StrOutputParser()

In [29]:
def rout(info):
    if 'positive' in info['sentiment'].lower():
        return positive_chain
    else:
        return negative_chain

In [30]:
# rout({'sentiment': 'negative'})

In [31]:
from langchain_core.runnables import RunnableLambda

full_chain = {"sentiment": chain, 'review': lambda x: x['review']} | RunnableLambda(rout)

In [32]:
full_chain

{
  sentiment: ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='Given the user review below, classify it as either being about `Positive` or `Negative`.\n            Do not respond with more than one word.\n\n            Review: {review}\n            Classification:'), additional_kwargs={})])
             | ChatOllama(model='qwen3:0.6b', base_url='http://localhost:11434')
             | StrOutputParser(),
  review: RunnableLambda(lambda x: x['review'])
}
| RunnableLambda(rout)

In [35]:
# review = "Thank you so much for the providing such a great plateform for learning. I am really happy with the service."
review = "I am not happy with the service. It is not good."

output = full_chain.invoke({'review': review})
print(output)

Subject: Follow-Up on Service Feedback  

Dear [User's Name],  

Thank you for your kind words. I sincerely apologize for the inconvenience and frustration you’ve had with the service. Your feedback is deeply appreciated, and I hope this message helps address your concerns.  

If you’d like to share your specific issue or provide further details, please feel free to reach out to me at [example@email.com]. I’m here to help resolve anything you may need.  

Looking forward to your response!  

Best regards,  
[Your Name]  

---  
This response acknowledges the user’s concern, invites further communication, and maintains a professional and empathetic tone.


## Make Custom Chain Runnables with RunnablePassthrough and RunnableLambda
- This is useful for formatting or when you need functionality not provided by other LangChain components, and custom functions used as Runnables are called RunnableLambdas.


In [36]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [38]:
def char_counts(text):
    return len(text)

def word_counts(text):
    return len(text.split())

prompt = ChatPromptTemplate.from_template("Explain these inputs in 5 sentences: {input1} and {input2}")

In [39]:
prompt

ChatPromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, template='Explain these inputs in 5 sentences: {input1} and {input2}'), additional_kwargs={})])

In [40]:
chain = prompt | llm | StrOutputParser()

output = chain.invoke({'input1': 'Earth is planet', 'input2': 'Sun is star'})

print(output)

1. Earth is a planet in our solar system, a terrestrial body that has sustained life through its unique environment.  
2. The Sun is a star, a massive celestial body that provides the energy necessary for life and the solar system's orbit.  
3. Earth's position in the solar system, located in the habitable zone, allows for liquid water and conditions suitable for life.  
4. As a planet, Earth has the ability to support complex ecosystems, making it a vital part of the solar system's structure.  
5. The Sun's role as a star is fundamental in sustaining the planet and the universe's energy flow.


In [41]:
chain = prompt | llm | StrOutputParser() | {'char_counts': RunnableLambda(char_counts), 
                                            'word_counts': RunnableLambda(word_counts), 
                                            'output': RunnablePassthrough()}

output = chain.invoke({'input1': 'Earth is planet', 'input2': 'Sun is star'})

print(output)

{'char_counts': 579, 'word_counts': 103, 'output': "1. Earth is a planet that supports life and has a unique atmosphere and geography.  \n2. The Sun is a star that provides the energy necessary for life and holds the solar system in balance.  \n3. Both Earth and the Sun are integral to our planet's environment, with Earth being home to humans and the Sun driving the cosmos.  \n4. They are part of the solar system and play critical roles in maintaining the balance of our planet and the universe.  \n5. The relationship between Earth and the Sun highlights their shared origin from a single star and their coexistence in the cosmos."}


## Custom Chain using ```@chain``` decorator

In [42]:
from langchain_core.runnables import chain

In [44]:
@chain
def custom_chain(params):
    return {
        'fact': fact_chain.invoke(params),
        'poem': poem_chain.invoke(params),
    }

params = {'school': 'primary', 'topics': 'solar system', 'points': 2, 'sentences': 2}
output = custom_chain.invoke(params)
print(output['fact'])
print('\n\n')
print(output['poem'])

1. The solar system consists of the Sun and eight planets, including Earth, orbiting around the Sun.  
2. Planets like Mercury and Venus orbit the Sun in the inner part of the solar system, while Earth is the only planet that has life.



Our sun, the heart of the cosmos,  
A world in space, a dance of stars.
